In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# 1. 스파크 세션 생성 (로컬 테스트용, 복잡한 외부 패키지 불필요)
spark = (
    SparkSession.builder.appName("ExtremeIngestionLocalTest")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")  # AQE 활성화
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.files.maxPartitionBytes", "134217728")  # 128MB 파티션
    .getOrCreate()
)

# 2. 테스트용 가짜 데이터(CSV)를 만들기 위한 경로 설정
local_dir = "./test_data"
os.makedirs(local_dir, exist_ok=True)
sample_csv_path = os.path.join(local_dir, "sample_logs.csv")

# 가상의 CSV 파일 생성 (실무에서는 이 위치에 수집할 파일들이 쌓임)
with open(sample_csv_path, "w", encoding="utf-8") as f:
    f.write("log_id,user_id,action_type,amount,event_time\n")
    f.write("LOG-001,101,click,15.5,2026-06-01 10:00:00\n")
    f.write("LOG-002,102,purchase,120.0,2026-06-01 11:30:00\n")
    f.write("LOG-003,101,view,5.0,2026-06-02 09:15:00\n")

# 3. [필수] 스키마 사전 정의 (inferSchema 차단으로 속도 극대화)
ingest_schema = StructType(
    [
        StructField("log_id", StringType(), False),
        StructField("user_id", IntegerType(), True),
        StructField("action_type", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("event_time", TimestampType(), True),
    ]
)

# 4. [고성능 수집 단계]
df_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")  # 스키마 추론 생략 (병목 차단)
    .schema(ingest_schema)
    .csv(local_dir)
)

print("--- [수집된 원본 데이터 확인] ---")
df_raw.show()

# 5. [가공 및 적재 최적화]
# 날짜별 파티션을 위한 컬럼 생성
df_refined = df_raw.withColumn(
    "event_date", df_raw["event_time"].cast("date")
)

# 6. [최종 저장] Parquet 포맷 + 날짜별 파티셔닝 적용 적재
target_output_path = "./optimized_output_logs"

(
    df_refined.write.mode("overwrite")
    .partitionBy("event_date")
    .option("compression", "snappy")
    .parquet(target_output_path)
)

print(
    f"대용량 수집 및 최적화 Parquet 적재 완료! 저장 경로: {target_output_path}"
)

--- [수집된 원본 데이터 확인] ---
+-------+-------+-----------+------+-------------------+
| log_id|user_id|action_type|amount|         event_time|
+-------+-------+-----------+------+-------------------+
|LOG-001|    101|      click|  15.5|2026-06-01 10:00:00|
|LOG-002|    102|   purchase| 120.0|2026-06-01 11:30:00|
|LOG-003|    101|       view|   5.0|2026-06-02 09:15:00|
+-------+-------+-----------+------+-------------------+

대용량 수집 및 최적화 Parquet 적재 완료! 저장 경로: ./optimized_output_logs


In [5]:
import pandas as pd
import numpy as np
import time
import os

# 1. 2만 행의 데이터 생성
data = {
    'id': range(50000),
    'category': np.random.choice(['A', 'B', 'C', 'D'], 50000),
    'value': np.random.randn(50000),
    'timestamp': pd.date_range(start='2026-01-01', periods=50000, freq='min')
}
df = pd.DataFrame(data)

# 2. 성능 측정 준비
output_dir = "performance_test"
os.makedirs(output_dir, exist_ok=True)

# 3. CSV 저장 테스트
start_csv = time.time()
df.to_csv(os.path.join(output_dir, "test_data.csv"), index=False)
end_csv = time.time()
csv_duration = end_csv - start_csv
csv_size = os.path.getsize(os.path.join(output_dir, "test_data.csv"))

# 4. Parquet 저장 테스트
start_parquet = time.time()
df.to_parquet(os.path.join(output_dir, "test_data.parquet"), index=False, compression='snappy')
end_parquet = time.time()
parquet_duration = end_parquet - start_parquet
parquet_size = os.path.getsize(os.path.join(output_dir, "test_data.parquet"))

# 5. 결과 요약
results = {
    "Format": ["CSV", "Parquet"],
    "Time (seconds)": [csv_duration, parquet_duration],
    "File Size (bytes)": [csv_size, parquet_size]
}
results_df = pd.DataFrame(results)
print(results_df)

    Format  Time (seconds)  File Size (bytes)
0      CSV        0.094173            2370677
1  Parquet        0.018876            1217028


In [3]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
from pyspark.sql import SparkSession

# 1. 스파크 세션 생성
spark = (
    SparkSession.builder.appName("MelonChartAnalysis")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# 2. 멜론 TOP 100 크롤링 (Requests + BeautifulSoup)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
url = "https://www.melon.com/chart/index.htm"
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

rank_list, title_list, artist_list = [], [], []

# 50위까지, 100위까지의 태그 선택자 파싱
for item in soup.select("tbody > tr"):
    try:
        rank = item.select_one("span.rank").text
        title = item.select_one("div.rank01 a").text
        artist = item.select_one("div.rank02 a").text

        rank_list.append(int(rank))
        title_list.append(title)
        artist_list.append(artist)
    except AttributeError:
        continue

# 판다스 DataFrame 생성 후 스파크 DataFrame으로 변환
pd_df = pd.DataFrame(
    {"rank": rank_list, "title": title_list, "artist": artist_list}
)
df_melon = spark.createDataFrame(pd_df)

# [핵심 수집] 즉시 Parquet 포맷으로 압축 저장 (작은 파일 방지 및 메타데이터 확보)
df_melon.write.mode("overwrite").parquet("./melon_top100.parquet")
print("멜론 TOP 100 수집 및 Parquet 적재 완료!")

멜론 TOP 100 수집 및 Parquet 적재 완료!


In [19]:
from pyspark.sql import functions as F

# 1. 저장해 둔 파켓(Parquet) 파일 읽어오기 (컬럼 프로젝션과 메타데이터 즉시 로드)
df_melon = spark.read.parquet("./melon_top100.parquet")

print("--- [1. 수집된 데이터 확인] ---")
df_melon.show(10, truncate=False)

# -------------------------------------------------------------------------
# 2. 집계(Aggregation) 성능 최적화 예시: 아티스트별 곡 수 집계
# -------------------------------------------------------------------------
# 스파크는 셔플 전에 각 노드에서 미리 집계(Map-side Aggregation)를 수행합니다.
print("--- [2. 아티스트별 TOP 100 수록 곡 수 집계] ---")
artist_counts = (
    df_melon.groupBy("artist")
    .agg(
        F.count("title").alias("song_count"),
        F.min("rank").alias("highest_rank"),  # 가장 높은 순위
    )
    .orderBy(F.desc("song_count"), F.asc("highest_rank"))
)

artist_counts.show(20)

# -------------------------------------------------------------------------
# 3. 조인(Join) 성능 극대화 예시: 브로드캐스트 조인(Broadcast Join) 활용
# -------------------------------------------------------------------------
# 만약 가상의 '기획사(Agency)' 정보가 담긴 작은 데이터프레임이 있다고 가정해 봅시다.
agency_data = [
    ("aespa", "SM Entertainment"),
    ("IVE (아이브)", "Starship Entertainment"),
    ("RESCENE (리센느)", "더뮤즈 엔터테인먼트"),
    # ... (실무에서는 이 테이블이 수십만 건 이상일 수 있음)
]
df_agency = spark.createDataFrame(agency_data, ["artist", "agency"])

df_melon_clean = df_melon.withColumn(
    "artist", F.trim(F.col("artist"))
)  # 앞뒤 공백 제거

# [핵심] 작은 테이블은 broadcast() 함수로 감싸주어 네트워크 셔플을 원천 차단합니다.
print("--- [3. 브로드캐스트 조인을 통한 기획사 매핑] ---")
df_joined = df_melon_clean.join(
    F.broadcast(df_agency), on="artist", how="left"  # 작은 테이블 브로드캐스트 강제
)

df_joined.select("rank", "title", "artist", "agency").show(30, truncate=False)

# 곡의 개수 집계까지는 완벽하나
# 브로드캐스트 조인에서 다소 아쉬운 결과가 나왔다

--- [1. 수집된 데이터 확인] ---
+----+-----------+------------------+
|rank|title      |artist            |
+----+-----------+------------------+
|1   |BiiiG      |BIGBANG (빅뱅)    |
|2   |LOVE ATTACK|RESCENE (리센느)  |
|3   |갑자기     |아이오아이 (I.O.I)|
|4   |REDRED     |CORTIS (코르티스) |
|5   |Pretty Girl|RESCENE (리센느)  |
|6   |LEMONADE   |aespa             |
|7   |Deja Vu    |RESCENE (리센느)  |
|8   |만찬가     |태연 (TAEYEON)    |
|9   |It′s Me    |아일릿(ILLIT)     |
|10  |BAD        |ATEEZ(에이티즈)   |
+----+-----------+------------------+
only showing top 10 rows

--- [2. 아티스트별 TOP 100 수록 곡 수 집계] ---
+----------------------+----------+------------+
|                artist|song_count|highest_rank|
+----------------------+----------+------------+
|            방탄소년단|         6|          41|
|      RESCENE (리센느)|         4|           2|
|Hearts2Hearts (하츠...|         4|          16|
|       DAY6 (데이식스)|         4|          23|
|          IVE (아이브)|         4|          30|
|                 aespa|         3|      

In [20]:
# 리센느와 아이브가 포함된 행만 필터링해서 원본 문자열 출력 (앞뒤로 |를 붙여 공백 확인)
df_melon.filter(
    F.col("artist").like("%RESCENE%") | F.col("artist").like("%IVE%")
).select(
    F.concat(F.lit("["), F.col("artist"), F.lit("]"))
).show(
    truncate=False
)

+--------------------+
|concat([, artist, ])|
+--------------------+
|[RESCENE (리센느)]  |
|[RESCENE (리센느)]  |
|[RESCENE (리센느)]  |
|[IVE (아이브)]      |
|[RESCENE (리센느)]  |
|[IVE (아이브)]      |
|[IVE (아이브)]      |
|[IVE (아이브)]      |
+--------------------+



In [21]:
# 예시: 멜론 아티스트 이름에서 대소문자 무시 및 정제 후 조인하기
# (만약 멜론에는 "RESCENE"으로만 나오고, agency_data에는 "RESCENE (리센느)"로 되어있다면 방향을 맞춰주어야 합니다.)

# 1. 멜론 데이터의 아티스트 이름을 대문자로 통일하고 공백 제거
df_melon_clean = df_melon.withColumn(
    "artist_key", F.upper(F.trim(F.col("artist")))
)

# 2. 기획사 데이터도 동일하게 키 값 생성
agency_data = [
    ("AESPA", "SM Entertainment"),
    ("IVE", "Starship Entertainment"),  # 괄호 제거하고 핵심 키워드만 입력
    ("RESCENE", "더뮤즈 엔터테인먼트"),
]
df_agency = spark.createDataFrame(agency_data, ["artist_key", "agency"])

# 3. 정제된 키(artist_key)를 기준으로 브로드캐스트 조인 수행
df_joined = df_melon_clean.join(
    F.broadcast(df_agency), on="artist_key", how="left"
)

# 결과 확인 (artist_key 대신 원래 artist를 보여주도록 선택)
df_joined.select(
    "rank", "title", F.col("artist").alias("original_artist"), "agency"
).filter(
    F.col("original_artist").like("%RESCENE%")
    | F.col("original_artist").like("%IVE%")
).show(
    30, truncate=False
)

# 여전히 브로드캐스트 조인에서 다소 아쉬운 결과가 나왔다
# 즉 특수 공백 문자의 가능성이 있다

+----+-----------+----------------+------+
|rank|title      |original_artist |agency|
+----+-----------+----------------+------+
|2   |LOVE ATTACK|RESCENE (리센느)|NULL  |
|5   |Pretty Girl|RESCENE (리센느)|NULL  |
|7   |Deja Vu    |RESCENE (리센느)|NULL  |
|30  |BANG BANG  |IVE (아이브)    |NULL  |
|52  |Runaway    |RESCENE (리센느)|NULL  |
|74  |REBEL HEART|IVE (아이브)    |NULL  |
|96  |I AM       |IVE (아이브)    |NULL  |
|100 |BLACKHOLE  |IVE (아이브)    |NULL  |
+----+-----------+----------------+------+



In [22]:
from pyspark.sql import functions as F

# 1. 멜론 데이터의 아티스트 이름에서 특수 공백 문자(\u00A0)나 보이지 않는 공백을 일반 공백으로 치환 후 앞뒤 공백 제거
df_melon_clean = df_melon.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 2. 기획사 데이터도 동일하게 괄호 앞의 특수 공백이나 형태를 맞춰서 정의
agency_data = [
    ("aespa", "SM Entertainment"),
    (
        "IVE (아이브)",
        "Starship Entertainment",
    ),  # 혹은 아래 정규식으로 괄호째 날려버려도 좋습니다.
    ("RESCENE (리센느)", "더뮤즈 엔터테인먼트"),
]

df_agency = spark.createDataFrame(agency_data, ["artist", "agency"])
df_agency_clean = df_agency.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 3. 정제된 키('artist_cleaned')를 기준으로 브로드캐스트 조인 수행
print("--- [브로드캐스트 조인 재시도] ---")
df_joined = df_melon_clean.join(
    F.broadcast(df_agency_clean.select("artist_cleaned", "agency")),
    on="artist_cleaned",
    how="left",
)

# 결과 확인 (원래 아티스트 이름과 매핑된 기획사 확인)
df_joined.select(
    "rank", "title", F.col("artist").alias("original_artist"), "agency"
).filter(
    F.col("original_artist").like("%RESCENE%")
    | F.col("original_artist").like("%IVE%")
).show(
    30, truncate=False
)

--- [브로드캐스트 조인 재시도] ---
+----+-----------+----------------+----------------------+
|rank|title      |original_artist |agency                |
+----+-----------+----------------+----------------------+
|2   |LOVE ATTACK|RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|5   |Pretty Girl|RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|7   |Deja Vu    |RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|30  |BANG BANG  |IVE (아이브)    |Starship Entertainment|
|52  |Runaway    |RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|74  |REBEL HEART|IVE (아이브)    |Starship Entertainment|
|96  |I AM       |IVE (아이브)    |Starship Entertainment|
|100 |BLACKHOLE  |IVE (아이브)    |Starship Entertainment|
+----+-----------+----------------+----------------------+



In [23]:
from pyspark.sql import functions as F

# 1. 기존에 정제해 둔 멜론 데이터프레임 활용 (앞서 만든 artist_cleaned 장착)
df_melon_clean = df_melon.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 2. [확장] 기획사 정보 + 멤버(Members) 정보가 포함된 마스터 데이터 정의
# 실무에서는 이런 데이터가 RDB나 S3의 작은 참조 테이블(Dimension Table)로 존재합니다.
extended_agency_data = [
    ("aespa", "SM Entertainment", "카리나,윈터,지젤,닝닝"),
    ("IVE (아이브)", "Starship Entertainment", "안유진,가을,레이,장원영,리즈,이서"),
    ("RESCENE (리센느)", "더뮤즈 엔터테인먼트", "원이,리브,미나미,제나,메이"),
]

df_master = spark.createDataFrame(
    extended_agency_data, ["artist", "agency", "members"]
)

# 마스터 데이터도 동일하게 공백 정제 키 생성
df_master_clean = df_master.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 3. [핵심] 브로드캐스트 조인 수행 (참조 테이블이 작으므로 네트워크 셔플 제로!)
df_fully_joined = df_melon_clean.join(
    F.broadcast(df_master_clean.select("artist_cleaned", "agency", "members")),
    on="artist_cleaned",
    how="left",
)

# 4. 결과 출력 (순위, 제목, 아티스트, 기획사, 멤버 확인)
print("--- [멜론 TOP 100 + 기획사 + 멤버 브로드캐스트 조인 결과] ---")
df_fully_joined.select(
    "rank", 
    "title", 
    F.col("artist").alias("original_artist"), 
    "agency", 
    "members"
).filter(
    F.col("original_artist").like("%RESCENE%") | 
    F.col("original_artist").like("%IVE%") | 
    F.col("original_artist").like("%aespa%")
).show(30, truncate=False)

--- [멜론 TOP 100 + 기획사 + 멤버 브로드캐스트 조인 결과] ---
+----+---------------------------------------------+----------------+----------------------+---------------------------------+
|rank|title                                        |original_artist |agency                |members                          |
+----+---------------------------------------------+----------------+----------------------+---------------------------------+
|2   |LOVE ATTACK                                  |RESCENE (리센느)|더뮤즈 엔터테인먼트   |원이,리브,미나미,제나,메이       |
|5   |Pretty Girl                                  |RESCENE (리센느)|더뮤즈 엔터테인먼트   |원이,리브,미나미,제나,메이       |
|6   |LEMONADE                                     |aespa           |SM Entertainment      |카리나,윈터,지젤,닝닝            |
|7   |Deja Vu                                      |RESCENE (리센느)|더뮤즈 엔터테인먼트   |원이,리브,미나미,제나,메이       |
|30  |BANG BANG                                    |IVE (아이브)    |Starship Entertainment|안유진,가을,레이,장원영,리즈,이서|
|45  |Whiplash                    

In [26]:
from pyspark.sql import functions as F

# 1. 조인된 결과에서 아티스트 이름을 original_artist로 명확히 고정하여 DataFrame 새로 저장
df_fully_joined_named = df_fully_joined.withColumnRenamed("artist", "original_artist")

# 2. 멤버 문자열을 배열로 쪼갠 뒤 각각의 행으로 펼치기 (explode)
df_exploded = df_fully_joined_named.withColumn(
    "member_array", F.split(F.col("members"), ",")
).withColumn(
    "individual_member", F.explode(F.col("member_array"))
)

# 3. 결과 확인
df_exploded.select("title", "original_artist", "individual_member").show(20, truncate=False)

# individual_member (관계형 데이터베이스의 1NF(제1정규형) 형태인 원자값(Atomic Value) 구조로 바뀐 것)

+-----------+----------------+-----------------+
|title      |original_artist |individual_member|
+-----------+----------------+-----------------+
|LOVE ATTACK|RESCENE (리센느)|원이             |
|LOVE ATTACK|RESCENE (리센느)|리브             |
|LOVE ATTACK|RESCENE (리센느)|미나미           |
|LOVE ATTACK|RESCENE (리센느)|제나             |
|LOVE ATTACK|RESCENE (리센느)|메이             |
|Pretty Girl|RESCENE (리센느)|원이             |
|Pretty Girl|RESCENE (리센느)|리브             |
|Pretty Girl|RESCENE (리센느)|미나미           |
|Pretty Girl|RESCENE (리센느)|제나             |
|Pretty Girl|RESCENE (리센느)|메이             |
|LEMONADE   |aespa           |카리나           |
|LEMONADE   |aespa           |윈터             |
|LEMONADE   |aespa           |지젤             |
|LEMONADE   |aespa           |닝닝             |
|Deja Vu    |RESCENE (리센느)|원이             |
|Deja Vu    |RESCENE (리센느)|리브             |
|Deja Vu    |RESCENE (리센느)|미나미           |
|Deja Vu    |RESCENE (리센느)|제나             |
|Deja Vu    |RESCENE (리센느)|메이             |
|BANG BAN

In [31]:
# 캐싱 (Cache / Persist) — 중복 연산의 지옥 탈출하기
# 스파크는 기본적으로 지연 연산(Lazy Evaluation) 특성이 있어서, 
# 액션(show(), count(), write() 등)이 실행될 때마다 앞에서부터 연산을 다시 수행
# 따라서 반복해서 사용하는 데이터프레임은 메모리에 올려두어야(Cache) 합니다.

from pyspark.storagelevel import StorageLevel

# [핵심] 쪼개지고 조인된 완성형 데이터프레임을 메모리에 캐싱 선언
# - .cache()는 메모리 전용(MEMORY_AND_DISK 등 옵션 조절 가능)
df_exploded.cache()

# 첫 번째 액션 실행 (이 시점에 메모리에 적재됨)
print("--- [첫 번째 액션: 전체 멤버 분포 조회] ---")
df_exploded.groupBy("agency").count().show()

# 두 번째 액션 실행 (메모리에서 즉시 읽어오므로 속도가 비약적으로 빨라짐!)
print("--- [두 번째 액션: 특정 기획사 멤버들의 곡 조회] ---")
df_exploded.filter(F.col("agency") == "Starship Entertainment").select("title", "individual_member").show(25)

# 캐시를 다 사용한 후에는 메모리를 확보하기 위해 df_exploded.unpersist()

--- [첫 번째 액션: 전체 멤버 분포 조회] ---
+--------------------+-----+
|              agency|count|
+--------------------+-----+
|Starship Entertai...|   24|
| 더뮤즈 엔터테인먼트|   20|
|    SM Entertainment|   12|
+--------------------+-----+

--- [두 번째 액션: 특정 기획사 멤버들의 곡 조회] ---
+-----------+-----------------+
|      title|individual_member|
+-----------+-----------------+
|  BANG BANG|           안유진|
|  BANG BANG|             가을|
|  BANG BANG|             레이|
|  BANG BANG|           장원영|
|  BANG BANG|             리즈|
|  BANG BANG|             이서|
|REBEL HEART|           안유진|
|REBEL HEART|             가을|
|REBEL HEART|             레이|
|REBEL HEART|           장원영|
|REBEL HEART|             리즈|
|REBEL HEART|             이서|
|       I AM|           안유진|
|       I AM|             가을|
|       I AM|             레이|
|       I AM|           장원영|
|       I AM|             리즈|
|       I AM|             이서|
|  BLACKHOLE|           안유진|
|  BLACKHOLE|             가을|
|  BLACKHOLE|             레이|
|  BLACKHOLE|      

In [32]:
# 파티셔닝 (Partitioning) — 조회 성능의 극대화

# 자주 조회하는 조건(예: 기획사별, 날짜별)에 따라 물리적으로 폴더를 나누어 저장

# 조건에 맞는 폴더만 쏙 골라서 읽어옵니다

# [핵심] 'agency(기획사)' 기준으로 물리적 폴더를 나누어 Parquet로 저장
target_partition_path = "./melon_partitioned_data"

(
    df_exploded.write
    .mode("overwrite")
    .partitionBy("agency")  # 기획사별 폴더 파티셔닝
    .parquet(target_partition_path)
)

print("기획사별 파티셔닝 적재 완료!")

기획사별 파티셔닝 적재 완료!


In [33]:
# 파티셔닝된 경로에서 데이터 읽기
df_loaded = spark.read.parquet(target_partition_path)

# [Partition Pruning 발생] 
# 전체 테라바이트급 데이터 중 'Starship Entertainment' 폴더만 정확히 콕 집어서 읽음!
df_loaded.filter(F.col("agency") == "Starship Entertainment").select("title", "individual_member").show(truncate=False)

+-----------+-----------------+
|title      |individual_member|
+-----------+-----------------+
|REBEL HEART|안유진           |
|REBEL HEART|가을             |
|REBEL HEART|레이             |
|REBEL HEART|장원영           |
|REBEL HEART|리즈             |
|REBEL HEART|이서             |
|I AM       |안유진           |
|I AM       |가을             |
|I AM       |레이             |
|I AM       |장원영           |
|I AM       |리즈             |
|I AM       |이서             |
|BLACKHOLE  |안유진           |
|BLACKHOLE  |가을             |
|BLACKHOLE  |레이             |
|BLACKHOLE  |장원영           |
|BLACKHOLE  |리즈             |
|BLACKHOLE  |이서             |
|BANG BANG  |안유진           |
|BANG BANG  |가을             |
+-----------+-----------------+
only showing top 20 rows



In [34]:
# 🔥 요약: 스파크 성능을 극대화하는 최종 체크리스트

# 수집 단계: inferSchema 금지, 스키마 사전 정의 및 즉시 Parquet 변환 (I/O 감소)

# 조인 단계: 작은 테이블은 무조건 F.broadcast()로 셔플 원천 차단

# 구조화 단계: explode를 통한 정규화로 분산 처리 극대화

# 반복 연산 단계: 재사용되는 데이터는 .cache()로 메모리에 고정

# 저장 및 조회 단계: 자주 조회하는 컬럼 기준으로 partitionBy 적용 (Full Scan 방지)

In [50]:
# 이제 1차로 수집한 Top 100 결과를 parquet 에 저장하고

# 2차로 수집한 Topp 100 결과와 비교하여 순위 변화를 확인

# 멜론 TOP 100 크롤링 (Requests + BeautifulSoup)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
url = "https://www.melon.com/chart/index.htm"
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

rank_list, title_list, artist_list = [], [], []

# 50위까지, 100위까지의 태그 선택자 파싱
for item in soup.select("tbody > tr"):
    try:
        rank = item.select_one("span.rank").text
        title = item.select_one("div.rank01 a").text
        artist = item.select_one("div.rank02 a").text

        rank_list.append(int(rank))
        title_list.append(title)
        artist_list.append(artist)
    except AttributeError:
        continue

# 판다스 DataFrame 생성 후 스파크 DataFrame으로 변환
pd_df2 = pd.DataFrame(
    {"rank": rank_list, "title": title_list, "artist": artist_list}
)
df_melon2 = spark.createDataFrame(pd_df2)

# [핵심 수집] 즉시 Parquet 포맷으로 압축 저장 (작은 파일 방지 및 메타데이터 확보)
df_melon2.write.mode("overwrite").parquet("./melon_2차_top100.parquet")
print("멜론 TOP 100 수집 및 Parquet 적재 완료!")

멜론 TOP 100 수집 및 Parquet 적재 완료!


In [51]:
pd_df[27:]

,rank,title,artist
27,28,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤)
28,29,Surfin' Boy,Red Velvet (레드벨벳)
29,30,BANG BANG,IVE (아이브)
30,31,Heavy Serenade,NMIXX
31,32,어제보다 슬픈 오늘,우디 (Woody)
...,...,...,...
95,96,I AM,IVE (아이브)
96,97,2.0,방탄소년단
97,98,OVERDRIVE,TWS (투어스)
98,99,FOCUS,Hearts2Hearts (하츠투하츠)


In [52]:
pd_df2[27:]

,rank,title,artist
27,28,BANG BANG,IVE (아이브)
28,29,어제보다 슬픈 오늘,우디 (Woody)
29,30,Heavy Serenade,NMIXX
30,31,너의 모든 순간,성시경
31,32,Pop Off Pop Off,KiiiKiii (키키)
...,...,...,...
95,96,I AM,IVE (아이브)
96,97,FOCUS,Hearts2Hearts (하츠투하츠)
97,98,BLACKHOLE,IVE (아이브)
98,99,OVERDRIVE,TWS (투어스)


In [53]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. 1차(오전 10시)와 2차(오후 12시)에 저장해 둔 파켓 파일 각각 읽기
# (컬럼 프로젝션이 적용되어 필요한 메타데이터만 즉시 로드됩니다)
df_10am = spark.read.parquet("./melon_top100.parquet").withColumn(
    "time_slot", F.lit("10:00")
)

df_12pm = spark.read.parquet("./melon_2차_top100.parquet").withColumn(
    "time_slot", F.lit("12:00")
)

# 2. 두 데이터프레임을 위아래로 결합 (Union)
df_combined = df_10am.unionByName(df_12pm)

# 3. 10시 데이터와 12시 데이터를 곡 제목(title)과 아티스트(artist) 기준으로 나란히 배치
df_10 = df_combined.filter(F.col("time_slot") == "10:00").select(
    F.col("title"), F.col("artist"), F.col("rank").alias("rank_10am")
)

df_12 = df_combined.filter(F.col("time_slot") == "12:00").select(
    F.col("title"), F.col("artist"), F.col("rank").alias("rank_12pm")
)

# 4. 조인하여 순위 비교 (두 시간대에 공존하는 곡 기준, 혹은 left join으로 진입/이탈 곡 파악)
# 여기서는 10시와 12시 모두 비교하기 위해 outer join을 사용하거나 inner join을 사용할 수 있습니다.
df_rank_compare = df_10.join(
    df_12, on=["title", "artist"], how="outer"  # 새로 진입하거나 차트 아웃된 곡까지 모두 보려면 outer
)

# 5. 순위 변동 폭 계산
# (10시 순위 - 12시 순위: 양수면 순위 상승, 음수면 하락)
df_rank_diff = (
    df_rank_compare.withColumn(
        "rank_change", F.col("rank_10am") - F.col("rank_12pm")
    )
    .withColumn(
        "status",
        F.when(F.col("rank_10am").isNull(), "NEW 진입")
        .when(F.col("rank_12pm").isNull(), "OUT 아웃")
        .when(F.col("rank_change") > 0, "▲ 상승")
        .when(F.col("rank_change") < 0, "▼ 하락")
        .otherwise("- 유지"),
    )
)

print("--- [오전 10시 vs 오후 12시 멜론 차트 순위 변동 리포트] ---")
df_rank_diff.select(
    "title", "artist", "rank_10am", "rank_12pm", "rank_change", "status"
).orderBy(
    F.coalesce(F.col("rank_12pm"), F.lit(999))
).show(30, truncate=False)

--- [오전 10시 vs 오후 12시 멜론 차트 순위 변동 리포트] ---
+--------------------------------------------+--------------------------+---------+---------+-----------+------+
|title                                       |artist                    |rank_10am|rank_12pm|rank_change|status|
+--------------------------------------------+--------------------------+---------+---------+-----------+------+
|BiiiG                                       |BIGBANG (빅뱅)            |1        |1        |0          |- 유지|
|LOVE ATTACK                                 |RESCENE (리센느)          |2        |2        |0          |- 유지|
|갑자기                                      |아이오아이 (I.O.I)        |3        |3        |0          |- 유지|
|REDRED                                      |CORTIS (코르티스)         |4        |4        |0          |- 유지|
|Pretty Girl                                 |RESCENE (리센느)          |5        |5        |0          |- 유지|
|LEMONADE                                    |aespa                     |6        |

In [54]:
# 1. 1차(오전 10시) Parquet 파일 읽어서 확인
print("=== [1차 파일: ./melon_top100.parquet] ===")
df_10 = spark.read.parquet("./melon_top100.parquet")
print(f"총 데이터 건수: {df_10.count()}개")
df_10.orderBy("rank").show(10, truncate=False)

# 2. 2차(오후 12시) Parquet 파일 읽어서 확인
print("=== [2차 파일: ./melon_2차_top100.parquet] ===")
df_12 = spark.read.parquet("./melon_2차_top100.parquet")
print(f"총 데이터 건수: {df_12.count()}개")
df_12.orderBy("rank").show(10, truncate=False)

=== [1차 파일: ./melon_top100.parquet] ===
총 데이터 건수: 100개
+----+-----------+------------------+
|rank|title      |artist            |
+----+-----------+------------------+
|1   |BiiiG      |BIGBANG (빅뱅)    |
|2   |LOVE ATTACK|RESCENE (리센느)  |
|3   |갑자기     |아이오아이 (I.O.I)|
|4   |REDRED     |CORTIS (코르티스) |
|5   |Pretty Girl|RESCENE (리센느)  |
|6   |LEMONADE   |aespa             |
|7   |Deja Vu    |RESCENE (리센느)  |
|8   |만찬가     |태연 (TAEYEON)    |
|9   |It′s Me    |아일릿(ILLIT)     |
|10  |BAD        |ATEEZ(에이티즈)   |
+----+-----------+------------------+
only showing top 10 rows

=== [2차 파일: ./melon_2차_top100.parquet] ===
총 데이터 건수: 100개
+----+-----------+------------------+
|rank|title      |artist            |
+----+-----------+------------------+
|1   |BiiiG      |BIGBANG (빅뱅)    |
|2   |LOVE ATTACK|RESCENE (리센느)  |
|3   |갑자기     |아이오아이 (I.O.I)|
|4   |REDRED     |CORTIS (코르티스) |
|5   |Pretty Girl|RESCENE (리센느)  |
|6   |LEMONADE   |aespa             |
|7   |It′s Me    |아일릿(ILLIT)     |
|8   |D

In [56]:
# 멜론 TOP 100 크롤링 (Requests + BeautifulSoup)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
url = "https://www.melon.com/chart/index.htm"
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

rank_list, title_list, artist_list, album_list = [], [], [], []

# 50위까지, 100위까지의 태그 선택자 파싱
for item in soup.select("tbody > tr"):
    try:
        rank = item.select_one("span.rank").text
        title = item.select_one("div.rank01 a").text
        artist = item.select_one("div.rank02 a").text
        # [확장] 앨범명 태그 선택자 추가 (보통 div.rank03 a 에 위치함)
        album = item.select_one("div.rank03 a").text

        rank_list.append(int(rank))
        title_list.append(title)
        artist_list.append(artist)
        album_list.append(album)
    except AttributeError:
        continue

# 판다스 DataFrame 생성 (album_name 컬럼 추가)
pd_df3 = pd.DataFrame(
    {
        "rank": rank_list, 
        "title": title_list, 
        "artist": artist_list,
        "album_name": album_list
    }
)
df_melon3 = spark.createDataFrame(pd_df3)

In [58]:
pd_df3

,rank,title,artist,album_name
0,1,BiiiG,BIGBANG (빅뱅),BiiiG
1,2,LOVE ATTACK,RESCENE (리센느),SCENEDROME
2,3,갑자기,아이오아이 (I.O.I),I.O.I 3rd MINI ALBUM [I.O.I : LOOP]
3,4,REDRED,CORTIS (코르티스),GREENGREEN
4,5,Pretty Girl,RESCENE (리센느),Pretty Girl - Special Single
...,...,...,...,...
95,96,I AM,IVE (아이브),I've IVE
96,97,FOCUS,Hearts2Hearts (하츠투하츠),FOCUS - The 1st Mini Album
97,98,BLACKHOLE,IVE (아이브),REVIVE+
98,99,OVERDRIVE,TWS (투어스),play hard


In [59]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 1. 메인 차트 페이지 크롤링
url = "https://www.melon.com/chart/index.htm"
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

rank_list, title_list, artist_list, album_list, date_list = [], [], [], [], []

print("멜론 TOP 100 메인 차트 파싱 및 상세 페이지(발매일) 수집 시작...")

for item in soup.select("tbody > tr"):
    try:
        # 곡 고유 ID 추출 (메인 테이블의 tr 태그 속성에 보통 data-song-no가 있음)
        song_id = item.get("data-song-no")
        rank = item.select_one("span.rank").text
        title = item.select_one("div.rank01 a").text
        artist = item.select_one("div.rank02 a").text
        album = item.select_one("div.rank03 a").text

        # 2. 각 곡의 상세 페이지에 접속하여 발매일(Release Date) 가져오기
        detail_url = f"https://www.melon.com/song/detail.htm?songId={song_id}"
        detail_res = requests.get(detail_url, headers=headers)
        detail_soup = BeautifulSoup(detail_res.text, "html.parser")

        # 멜론 상세 페이지의 발매일 영역 선택자 (meta 정보가 담긴 dl.list 또는 dd)
        # 보통 정보 영역에서 '발매일' 타이틀 다음의 텍스트를 파싱합니다.
        release_date = "정보 없음"
        meta_dl = detail_soup.select_one("div.section_info")
        if meta_dl:
            for dt in meta_dl.select("dt"):
                if "발매일" in dt.text:
                    # 바로 다음 형제 태그인 dd의 텍스트 가져오기
                    dd = dt.find_next_sibling("dd")
                    if dd:
                        release_date = dd.text.strip()
                        break

        # 리스트에 담기
        rank_list.append(int(rank))
        title_list.append(title)
        artist_list.append(artist)
        album_list.append(album)
        date_list.append(release_date)

        # 서버 차단 방지를 위한 미세한 딜레이 (0.1초)
        time.sleep(0.1)

    except Exception as e:
        # 에러가 난 곡은 건너뜀
        continue

# 3. 판다스 DataFrame으로 변환 후 스파크 DF로 적재
pd_df = pd.DataFrame({
    "rank": rank_list,
    "title": title_list,
    "artist": artist_list,
    "album_name": album_list,
    "release_date": date_list
})

df_melon_final = spark.createDataFrame(pd_df)

멜론 TOP 100 메인 차트 파싱 및 상세 페이지(발매일) 수집 시작...


In [60]:
pd_df

,rank,title,artist,album_name,release_date
0,1,BiiiG,BIGBANG (빅뱅),BiiiG,2026.08.19
1,2,LOVE ATTACK,RESCENE (리센느),SCENEDROME,2024.08.27
2,3,갑자기,아이오아이 (I.O.I),I.O.I 3rd MINI ALBUM [I.O.I : LOOP],2026.05.19
3,4,REDRED,CORTIS (코르티스),GREENGREEN,2026.04.20
4,5,Pretty Girl,RESCENE (리센느),Pretty Girl - Special Single,2026.07.08
...,...,...,...,...,...
95,96,I AM,IVE (아이브),I've IVE,2023.04.10
96,97,FOCUS,Hearts2Hearts (하츠투하츠),FOCUS - The 1st Mini Album,2025.10.20
97,98,BLACKHOLE,IVE (아이브),REVIVE+,2026.02.23
98,99,OVERDRIVE,TWS (투어스),play hard,2025.10.13


In [63]:
# Parquet 파일로 저장
target_path = "./melon_2차_top100_with_date.parquet"
df_melon_final.write.mode("overwrite").parquet(target_path)

In [68]:
# 1. 저장된 파켓 파일 읽기 (또는 메모리에 있는 df_melon_final 그대로 사용)
df_melon_final = spark.read.parquet("./melon_2차_top100_with_date.parquet")

# 2. 데이터 건수 및 스키마 구조 확인
print(f"--- [데이터 총 건수: {df_melon_final.count()}개] ---")
df_melon_final.printSchema()

# 3. 상위 10개 데이터 조회 (순위, 제목, 가수, 앨범, 발매일)
print("--- [TOP 10 멜론 차트 & 상세 정보 리포트] ---")
df_melon_final.orderBy("rank").show(10, truncate=False)

--- [데이터 총 건수: 100개] ---
root
 |-- rank: long (nullable = true)
 |-- title: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- release_date: string (nullable = true)

--- [TOP 10 멜론 차트 & 상세 정보 리포트] ---
+----+-----------+------------------+-----------------------------------+------------+
|rank|title      |artist            |album_name                         |release_date|
+----+-----------+------------------+-----------------------------------+------------+
|1   |BiiiG      |BIGBANG (빅뱅)    |BiiiG                              |2026.08.19  |
|2   |LOVE ATTACK|RESCENE (리센느)  |SCENEDROME                         |2024.08.27  |
|3   |갑자기     |아이오아이 (I.O.I)|I.O.I 3rd MINI ALBUM [I.O.I : LOOP]|2026.05.19  |
|4   |REDRED     |CORTIS (코르티스) |GREENGREEN                         |2026.04.20  |
|5   |Pretty Girl|RESCENE (리센느)  |Pretty Girl - Special Single       |2026.07.08  |
|6   |LEMONADE   |aespa             |LEMONADE - The 2nd Album  

In [69]:
from pyspark.sql import functions as F

# 1. '아이브' 또는 'IVE'가 포함된 아티스트 필터링
# (대소문자 무시 검색을 원하면 lower()를 쓸 수도 있지만, 보통 contains로 충분합니다)
df_ive = df_melon_final.filter(
    F.col("artist").contains("아이브") | F.col("artist").contains("IVE")
)

# 2. 결과 건수 확인
print(f"--- [아이브 차트 진입 곡 수: {df_ive.count()}개] ---")

# 3. 순위 순으로 정렬해서 리포트 출력
df_ive.orderBy("rank").show(truncate=False)

--- [아이브 차트 진입 곡 수: 4개] ---
+----+-----------+------------+-----------+------------+
|rank|title      |artist      |album_name |release_date|
+----+-----------+------------+-----------+------------+
|28  |BANG BANG  |IVE (아이브)|REVIVE+    |2026.02.09  |
|72  |REBEL HEART|IVE (아이브)|IVE EMPATHY|2025.01.13  |
|96  |I AM       |IVE (아이브)|I've IVE   |2023.04.10  |
|98  |BLACKHOLE  |IVE (아이브)|REVIVE+    |2026.02.23  |
+----+-----------+------------+-----------+------------+



In [7]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# 1. 스파크 세션 생성
spark = (
    SparkSession.builder.appName("MelonChartAnalysis")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://www.melon.com/chart/index.htm"
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

rank_list, title_list, artist_list, album_list, date_list, like_list, yesterday_rank_list = [], [], [], [], [], [], []

print("멜론 TOP 100 메인 및 상세 페이지 파싱 시작 (발매일, 좋아요, 어제 순위)...")

for item in soup.select("tbody > tr"):
    try:
        song_id = item.get("data-song-no")
        rank = item.select_one("span.rank").text
        title = item.select_one("div.rank01 a").text
        artist = item.select_one("div.rank02 a").text
        album = item.select_one("div.rank03 a").text

        # 상세 페이지 접속
        detail_url = f"https://www.melon.com/song/detail.htm?songId={song_id}"
        detail_res = requests.get(detail_url, headers=headers)
        detail_soup = BeautifulSoup(detail_res.text, "html.parser")

        # 1. 발매일 파싱 (<div class="meta"> 내부의 <dl class="list"> 탐색)
        release_date = "정보 없음"
        meta_dl = detail_soup.select_one("div.meta dl.list")
        if meta_dl:
            for dt, dd in zip(meta_dl.select("dt"), meta_dl.select("dd")):
                if "발매일" in dt.text:
                    release_date = dd.text.strip()
                    break

        # 2. 좋아요 수 파싱 (공백 및 줄바꿈 처리 추가)
        like_count = 0
        like_elem = detail_soup.select_one("#d_like_count")
        if like_elem:
            # 텍스트 내의 쉼표, 줄바꿈, 공백을 모두 제거
            like_text = like_elem.text.replace(",", "").replace("\n", "").strip()
            if like_text.isdigit():
                like_count = int(like_text)

        # 3. 어제의 차트 순위 파싱 (이미 성공함!)
        yesterday_rank = None  # 999 대신 None (NULL) 부여 (대신에 소수점 단위로 저장됨)
        chart_div = detail_soup.select_one("div.share div.chart")
        if chart_div:
            num_elem = chart_div.select_one("span.num")
            if num_elem and num_elem.text.strip().isdigit():
                yesterday_rank = int(num_elem.text.strip())

        # 데이터 리스트에 담기
        rank_list.append(int(rank))
        title_list.append(title)
        artist_list.append(artist)
        album_list.append(album)
        date_list.append(release_date)
        like_list.append(like_count)
        yesterday_rank_list.append(yesterday_rank)

        # 서버 부하 방지 딜레이
        time.sleep(0.1)

    except Exception as e:
        continue

# 판다스 DataFrame 생성 후 스파크 DF로 변환
pd_df = pd.DataFrame({
    "rank": rank_list,
    "title": title_list,
    "artist": artist_list,
    "album_name": album_list,
    "release_date": date_list,
    "likes": like_list,
    "yesterday_rank": yesterday_rank_list
})

df_melon_pro = spark.createDataFrame(pd_df)

# Parquet 파일로 저장
target_path = "./melon_pro_data.parquet"

# mode를 "overwrite"에서 "append"로 변경
# (만약 컬럼 순서나 미세한 스키마 차이가 걱정된다면 .option("mergeSchema", "true")를 함께 써주면 안전합니다)
df_melon_pro.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .parquet(target_path)

print("기존 데이터에 새로운 데이터 Append 적재 완료!")
df_melon_pro.orderBy("rank").show(10, truncate=False)

# 안타깝게도 좋아요 는 상세 페이지에서 확인이 불가능함

멜론 TOP 100 메인 및 상세 페이지 파싱 시작 (발매일, 좋아요, 어제 순위)...
기존 데이터에 새로운 데이터 Append 적재 완료!
+----+-----------+------------------+-----------------------------------+------------+-----+--------------+
|rank|title      |artist            |album_name                         |release_date|likes|yesterday_rank|
+----+-----------+------------------+-----------------------------------+------------+-----+--------------+
|1   |BiiiG      |BIGBANG (빅뱅)    |BiiiG                              |2026.08.19  |0    |NaN           |
|2   |LOVE ATTACK|RESCENE (리센느)  |SCENEDROME                         |2024.08.27  |0    |1.0           |
|3   |갑자기     |아이오아이 (I.O.I)|I.O.I 3rd MINI ALBUM [I.O.I : LOOP]|2026.05.19  |0    |2.0           |
|4   |REDRED     |CORTIS (코르티스) |GREENGREEN                         |2026.04.20  |0    |3.0           |
|5   |Pretty Girl|RESCENE (리센느)  |Pretty Girl - Special Single       |2026.07.08  |0    |5.0           |
|6   |LEMONADE   |aespa             |LEMONADE - The 2nd Album           |202

In [6]:
import requests
from bs4 import BeautifulSoup

# 테스트용 곡 ID (예: 방금 보신 BANG BANG의 ID)
test_song_id = "601237102"
detail_url = f"https://www.melon.com/song/detail.htm?songId={test_song_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

res = requests.get(detail_url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

# 좋아요 요소 직접 찾기 테스트
like_elem = soup.select_one("#d_like_count")
if like_elem:
    print(f"찾은 좋아요 텍스트: [{like_elem.text}]")
else:
    print("❌ #d_like_count 태그를 아예 찾지 못했습니다!")

# 어제 순위 요소 직접 찾기 테스트
chart_div = soup.select_one("div.share div.chart")
if chart_div:
    print(f"찾은 어제 순위 HTML: [{chart_div}]")
else:
    print("❌ div.share div.chart 태그를 찾지 못했습니다!")

찾은 좋아요 텍스트: [
총건수
											0
										]
찾은 어제 순위 HTML: [<div class="chart">
<span class="bullet_icons chart_i"></span> 어제의 차트 순위 <span class="num">26</span>위
							</div>]


In [2]:
!pip install playwright
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 56.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.6/624.6 kB 94.5 MB/s eta 0:00:00
  Attempting uninstall: greenlet
    Found existing installation: greenlet 3.0.0
    Uninstalling greenlet-3.0.0:
      Successfully uninstalled greenlet-3.0.0
184.3 MiB [                    ] 0% 294.6s184.3 MiB [                    ] 0% 466.5s184.3 MiB [                    ] 0% 572.2s184.3 MiB [                    ] 0% 234.6s184.3 MiB [                    ] 0% 129.8s184.3 MiB [                    ] 0% 93.1s184.3 MiB [                    ] 0% 35.1s184.3 MiB [                    ] 2% 16.5s184.3 MiB [=                   ] 2% 12.6s184.3 MiB [=                   ] 3% 10.8s184.3 MiB [=                   ] 4% 10.8s184.3 MiB [=                   ] 4% 9.9s184.3 MiB [=                   ] 5% 9.0s184.3 MiB [=                   ] 6% 8.4s184.3 MiB [=                   ] 6% 8.9s184.3 MiB [=                   ] 6% 8.6s184.3 MiB [= 

In [3]:
!playwright install-deps

# 설치를 위해 docker 를 관리자 권한으로 open 해야 함

# docker run -it --rm --user root -e GRANT_SUDO=yes  -p 8888:8888   -p 4040:4040   -v "$(pwd)/work:/home/jovyan/work"   jupyter/pyspark-notebook:latest

Installing dependencies...
Switching to root user to install dependencies...
Get:1 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [84.1 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,314 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,569 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 Packages [266 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy/main amd64 Packages [1,792 kB]   
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,203 kB]
Get:12 http://archive.ubuntu.com/ubuntu

In [6]:
from pyspark.sql import SparkSession

# 스파크 세션 생성
spark = (
    SparkSession.builder.appName("MelonChartAnalysis")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In [22]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd

async def get_melon_chart_with_likes():
    async with async_playwright() as p:
        # 헤드리스 브라우저 실행
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        
        print("멜론 차트 페이지 접속 중...")
        await page.goto("https://www.melon.com/chart/index.htm")
        
        # 좋아요 등 자바스크립트 데이터가 렌더링될 때까지 충분히 대기
        await page.wait_for_selector("tbody > tr", timeout=10000)
        await asyncio.sleep(2) 
        
        rank_list, title_list, artist_list, album_list, like_list = [], [], [], [], []
        
        # 페이지 내의 행(tr)들 가져오기
        rows = await page.locator("tbody > tr").all()
        
        for idx, item in enumerate(rows):
            try:
                # 1. 순위
                rank_elem = item.locator("span.rank")
                rank = await rank_elem.inner_text() if await rank_elem.count() > 0 else str(idx + 1)
                
                # 2. 제목 (.first 추가로 중복 방지)
                title_elem = item.locator("div.rank01 a").first
                title = await title_elem.inner_text() if await title_elem.count() > 0 else ""
                
                # 3. 가수 (.first 추가로 중복 방지)
                artist_elem = item.locator("div.rank02 a").first
                artist = await artist_elem.inner_text() if await artist_elem.count() > 0 else ""
                
                # 4. 앨범명 (.first 추가로 중복 방지)
                album_elem = item.locator("div.rank03 a").first
                album = await album_elem.inner_text() if await album_elem.count() > 0 else ""
                
                # 5. 좋아요 수
                like_count = 0
                like_elem = item.locator("button.like span.cnt")
                if await like_elem.count() > 0:
                    raw_text = await like_elem.inner_text()
                    clean_text = raw_text.replace("총건수", "").replace(",", "").strip()
                    if clean_text.isdigit():
                        like_count = int(clean_text)
                
                rank_list.append(int(rank.strip()))
                title_list.append(title.strip())
                artist_list.append(artist.strip())
                album_list.append(album.strip())
                like_list.append(like_count)
                
            except Exception as e:
                print(f"[{idx+1}번째 행 파싱 에러 발생]: {e}")
                continue

        print("수집된 데이터 개수:", len(rank_list))
                
        await browser.close()
        
        # 판다스 및 스파크 DF 변환
        pd_df = pd.DataFrame({
            "rank": rank_list,
            "title": title_list,
            "artist": artist_list,
            "album_name": album_list,
            "likes": like_list
        })

        print(pd_df)
        
        return spark.createDataFrame(pd_df)

# 코딩 환경에 맞춰 실행 (주피터 노트북인 경우 await 사용 가능)
df_melon_likes = await get_melon_chart_with_likes()

# 파켓 저장
df_melon_likes.write.mode("overwrite").parquet("./melon_firepower_data.parquet")
print("아이브 화력 분석용 데이터 수집 완료!")
df_melon_likes.filter(df_melon_likes["artist"].contains("아이브")).show(truncate=False)

멜론 차트 페이지 접속 중...
수집된 데이터 개수: 100
    rank        title         artist                           album_name  \
0      1        BiiiG   BIGBANG (빅뱅)                                BiiiG   
1      2  LOVE ATTACK  RESCENE (리센느)                           SCENEDROME   
2      3          갑자기  아이오아이 (I.O.I)  I.O.I 3rd MINI ALBUM [I.O.I : LOOP]   
3      4       REDRED  CORTIS (코르티스)                           GREENGREEN   
4      5  Pretty Girl  RESCENE (리센느)         Pretty Girl - Special Single   
..   ...          ...            ...                                  ...   
95    96    OVERDRIVE      TWS (투어스)                            play hard   
96    97     우리들의 블루스            임영웅                              IM HERO   
97    98    BLACKHOLE      IVE (아이브)                              REVIVE+   
98    99     Hooligan          방탄소년단                              ARIRANG   
99   100          2.0          방탄소년단                              ARIRANG   

     likes  
0    29550  
1   138039  
2 

In [23]:
df_melon_likes.show()

+----+-------------------------+----------------------+--------------------+------+
|rank|                    title|                artist|          album_name| likes|
+----+-------------------------+----------------------+--------------------+------+
|   1|                    BiiiG|        BIGBANG (빅뱅)|               BiiiG| 29550|
|   2|              LOVE ATTACK|      RESCENE (리센느)|          SCENEDROME|138039|
|   3|                   갑자기|    아이오아이 (I.O.I)|I.O.I 3rd MINI AL...| 74833|
|   4|                   REDRED|     CORTIS (코르티스)|          GREENGREEN| 84802|
|   5|              Pretty Girl|      RESCENE (리센느)|Pretty Girl - Spe...| 54168|
|   6|                 LEMONADE|                 aespa|LEMONADE - The 2n...| 55222|
|   7|                  Deja Vu|      RESCENE (리센느)|             Dearest| 45382|
|   8|                  It′s Me|         아일릿(ILLIT)|    MAMIHLAPINATAPAI| 55496|
|   9|                   만찬가|        태연 (TAEYEON)|  J-POP REMAKE Vol.1| 38859|
|  10|                 